### GOLD TESTING - FACT TRANSACTIONS (SCD1)

#### Purpose
- Validate `coffee.gold.fact_transactions` against `coffee.silver.transactions`
- Ensure the Gold fact table contains correct measures and keys
- Ensure no duplicate transactions exist
- Validate referential integrity from `fact_transactions` to dimension tables

#### Tests Covered
1. Silver vs Gold row count reconciliation
2. Null checks on primary key (`transaction_id`)
3. Duplicate check on `transaction_id`
4. Amount reconciliation (final_amount, original_amount, discount_applied)
5. Referential integrity:
   - store_id → dim_stores
   - payment_method_id → dim_payment_methods
   - voucher_id → dim_vouchers (only for non-null voucher_id)
   - user_id → dim_users (only for non-null user_id)


In [0]:
-- TEST 1: SILVER vs GOLD ROW COUNT RECONCILIATION
SELECT
  'fact_transactions_count_recon' AS test_name,
  (SELECT COUNT(*) FROM coffee.silver.transactions) AS silver_count,
  (SELECT COUNT(*) FROM coffee.gold.fact_transactions) AS gold_count;

In [0]:
-- TEST 2: NULL CHECK ON PRIMARY KEY
SELECT
  'fact_transactions_null_transaction_id' AS test_name,
  COUNT(*) AS null_key_count
FROM coffee.gold.fact_transactions
WHERE transaction_id IS NULL;


In [0]:
-- TEST 3: DUPLICATE CHECK ON PRIMARY KEY
SELECT
  'fact_transactions_duplicate_transaction_id' AS test_name,
  COUNT(*) AS duplicate_key_count
FROM (
  SELECT transaction_id
  FROM coffee.gold.fact_transactions
  GROUP BY transaction_id
  HAVING COUNT(*) > 1
);

In [0]:
-- TEST 4: AMOUNT RECONCILIATION (SILVER vs GOLD)
-- This ensures measures were not altered during transformation.
SELECT
  'transactions_final_amount' AS metric,
  (SELECT ROUND(SUM(final_amount), 2) FROM coffee.silver.transactions) AS silver_sum,
  (SELECT ROUND(SUM(final_amount), 2) FROM coffee.gold.fact_transactions) AS gold_sum

UNION ALL
SELECT
  'transactions_original_amount' AS metric,
  (SELECT ROUND(SUM(original_amount), 2) FROM coffee.silver.transactions) AS silver_sum,
  (SELECT ROUND(SUM(original_amount), 2) FROM coffee.gold.fact_transactions) AS gold_sum

UNION ALL
SELECT
  'transactions_discount_applied' AS metric,
  (SELECT ROUND(SUM(discount_applied), 2) FROM coffee.silver.transactions) AS silver_sum,
  (SELECT ROUND(SUM(discount_applied), 2) FROM coffee.gold.fact_transactions) AS gold_sum;



In [0]:
-- TEST 5: REFERENTIAL INTEGRITY - FACT TRANSACTIONS -> DIM STORES
SELECT
  'RI_fact_transactions_store_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transactions f
LEFT JOIN coffee.gold.dim_stores d
  ON f.store_id = d.store_id AND d.__END_AT IS NULL
WHERE d.store_id IS NULL;

In [0]:
-- TEST 6: REFERENTIAL INTEGRITY - FACT TRANSACTIONS -> DIM PAYMENT METHODS
SELECT
  'RI_fact_transactions_payment_method_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transactions f
LEFT JOIN coffee.gold.dim_payment_methods d
  ON f.payment_method_id = d.method_id AND d.__END_AT IS NULL
WHERE d.method_id IS NULL;

In [0]:
-- TEST 7: REFERENTIAL INTEGRITY - FACT TRANSACTIONS -> DIM VOUCHERS
-- voucher_id can be NULL, so only validate non-null voucher_id values
SELECT
  'RI_fact_transactions_voucher_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transactions f
LEFT JOIN coffee.gold.dim_vouchers d
  ON f.voucher_id = d.voucher_id AND d.__END_AT IS NULL
WHERE f.voucher_id IS NOT NULL
  AND d.voucher_id IS NULL;

In [0]:
-- TEST 8: REFERENTIAL INTEGRITY - FACT TRANSACTIONS -> DIM USERS
-- user_id can be NULL (guest transactions), so only validate non-null user_id values
SELECT
  'RI_fact_transactions_user_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transactions f
LEFT JOIN coffee.gold.dim_users d
  ON f.user_id = d.user_id AND d.__END_AT IS NULL
WHERE f.user_id IS NOT NULL
  AND d.user_id IS NULL;